# Explanations: what the app will say about each lounge and area

Each decision (`decisions.ipynb`) gets a pyramid: a one-sentence answer, then the arguments that
matter most for it (ranked by `src.explain.prioritize`, never by the writer), each backed by 2-5
numbers copied from the decision's fact sheet. `src.explain.verify` rejects any explanation that
cites a value or writes a number not on the fact sheet.

The **AI** explanations here were written in a Claude Code session from the same prompts
`just explain` sends to the API (`python -m src.explain prompts|check|ingest`), so no API credits
were spent; they are cached in `data/explanations/cache.json` by a hash of their fact sheet, so a
changed number falls back to the deterministic **template**. The NOT SCORED airport lounge and
SKIP areas always use the template.

Data: `src.explain.v3_subjects()` (baseline, medium levels).

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from collections import Counter

import pandas as pd
from IPython.display import Markdown, display

from src.explain import (LOUNGE_TABLE, AREA_TABLE, explain, load_cache, table_rows,
                         v3_subjects, verify)

pd.set_option("display.max_colwidth", 120)
cache = load_cache()
subjects = v3_subjects()
exps = {(k, sid): explain(k, sid, a, f, cache=cache) for k, sid, a, f in subjects}
facts = {(k, sid): f for k, sid, a, f in subjects}
print(Counter((k, e.source) for (k, _), e in exps.items()))
print("grounding failures:", sum(bool(verify(e, facts[key], key[0])) for key, e in exps.items()))

Counter({('area', 'template'): 372, ('area', 'ai'): 41, ('lounge', 'ai'): 23, ('uae', 'ai'): 1, ('lounge', 'template'): 1})
grounding failures: 0


In [2]:
def show(kind, sid):
    e, f = exps[(kind, sid)], facts[(kind, sid)]
    lines = [f"### {sid} — **{e.action}** ({e.source})", "", f"**{e.headline}**", ""]
    for r in e.reasons:
        ev = "; ".join(f"{x.label}: {x.value}" for x in r.evidence)
        lines += [f"- **{r.topic}**: {r.claim}  \n  <small>{ev}</small>"]
    lines += ["", f"_{e.table_caption}_"]
    display(Markdown("\n".join(lines)))


def table(kind, sid):
    fields = {"lounge": LOUNGE_TABLE, "area": AREA_TABLE}[kind]
    return pd.DataFrame(table_rows(fields, facts[(kind, sid)])).set_index("Factor")

## The network summary (the app opens with this)

In [3]:
show("uae", "uae")

### uae — **SUMMARY** (ai)

**Of 23 scored lounges, 3 are PROTECT, 15 HOLD and 5 SHRINK, and 5 of 413 growth areas pass both tests to GROW.**

- **shrink**: 5 lounges are SHRINK, to investigate before their lease events: delma, khaleej-al-arabi, mohammed-bin-zayed-city, noya-plaza and westyas, the strongest at 0.28.  
  <small>SHRINK: 5; SHRINK lounges: delma, khaleej-al-arabi, mohammed-bin-zayed-city, noya-plaza, westyas; Strongest SHRINK composite: 0.2773</small>
- **grow**: 5 areas are GROW, with about 262,000 women 15+ between them, 9.7 to 43.8 km from the nearest lounge.  
  <small>GROW: 5; GROW areas: al-dhaid-sharjah, al-jerf-ajman, kalba-sharjah, khor-fakkan-sharjah, sharjah-sharjah; Women in GROW areas: 261797; Closest GROW area to a lounge: 9.6915; Furthest GROW area from a lounge: 43.7824</small>
- **protect**: 3 lounges are PROTECT: al-ain, al-taif-mall and ras-al-khaimah, the weakest at 0.68.  
  <small>PROTECT: 3; PROTECT lounges: al-ain, al-taif-mall, ras-al-khaimah; Weakest PROTECT composite: 0.6762</small>
- **confidence**: 15 lounges sit in the HOLD band and 10 of the 23 calls are low confidence.  
  <small>HOLD: 15; Low-confidence calls: 10; Lounges scored: 23</small>
- **data_gaps**: There is no revenue or rent data, so these are location and market calls from 3,833 searched salons, not return-on-capital calls.  
  <small>Revenue data: False; Rent data: False; Salons searched: 3833</small>

_Lounge calls come from a fixed-scale scorecard of demand, cannibalisation, capture and rating; growth-area calls from two tests, size and premium saturation. Counts cover the 23 scored lounges; the airport lounge is not scored._

## Lounges, PROTECT first

In [4]:
order = {"PROTECT": 0, "HOLD": 1, "SHRINK": 2, "NOT SCORED": 3}
for k, sid, a, f in sorted((s for s in subjects if s[0] == "lounge"), key=lambda s: (order[s[2]], s[1])):
    show(k, sid)

### al-ain — **PROTECT** (ai)

**PROTECT: Al Ain shares none of its 102,000 catchment women with a sibling and holds 12% of premium reviews, for a composite of 0.77.**

- **cannibalisation**: No other Bedashing lounge reaches any of its catchment women (0% shared).  
  <small>Shared catchment: 0.0; Cannibalisation score: 1.0; Cannibalisation contribution: 0.2857</small>
- **capture**: It holds 12% of premium reviews against 16 premium substitutes in a pool of 59.  
  <small>Capture: 0.1232; Premium substitutes: 16; Premium salons in catchment: 59; Capture score: 0.8211</small>
- **rating**: Its 4.7 rating is 0.15 above the 4.55 median of the premium salons in its catchment.  
  <small>Google rating: 4.7; Premium salons' median rating: 4.55; Rating gap: 0.15</small>

_Each signal is scored 0-1 on a fixed scale where 1 is good for the lounge, and the contribution rows add up to the composite. Premium saturation (premium reviews per 1k women in the catchment) is shown for context and is not scored._

### al-taif-mall — **PROTECT** (ai)

**PROTECT, low confidence: Al Taif Mall shares none of its 44,800 women with a sibling and holds 14% of premium reviews, but at 0.69 it is 0.04 above the HOLD line.**

- **cannibalisation**: No other Bedashing lounge reaches any of its catchment women (0% shared).  
  <small>Shared catchment: 0.0; Cannibalisation score: 1.0; Cannibalisation contribution: 0.2857</small>
- **capture**: It holds 14% of premium reviews against 7 substitutes in a pool of 35 premium salons.  
  <small>Capture: 0.1428; Premium substitutes: 7; Premium salons in catchment: 35; Capture score: 0.9523</small>

_Each signal is scored 0-1 on a fixed scale where 1 is good for the lounge, and the contribution rows add up to the composite. Premium saturation (premium reviews per 1k women in the catchment) is shown for context and is not scored._

### ras-al-khaimah — **PROTECT** (ai)

**PROTECT, low confidence: Ras Al Khaimah shares none of its 55,800 women with a sibling and holds 14% of premium reviews, but at 0.68 it is 0.03 above the HOLD line.**

- **cannibalisation**: No other Bedashing lounge reaches any of its catchment women (0% shared).  
  <small>Shared catchment: 0.0; Cannibalisation score: 1.0; Cannibalisation contribution: 0.2857</small>
- **capture**: It holds 14% of premium reviews against 8 substitutes in a pool of 44 premium salons.  
  <small>Capture: 0.1381; Premium substitutes: 8; Premium salons in catchment: 44; Capture score: 0.9209</small>

_Each signal is scored 0-1 on a fixed scale where 1 is good for the lounge, and the contribution rows add up to the composite. Premium saturation (premium reviews per 1k women in the catchment) is shown for context and is not scored._

### al-barsha — **HOLD** (ai)

**HOLD, low confidence: Al Barsha's 321,000 addressable women keep it 0.02 above the SHRINK line, despite holding about 1% of premium reviews.**

- **demand**: Demand is at the top of the scale: 321,000 addressable women, women-weighted rent AED 115,000, rents observed for 99% of them.  
  <small>Addressable women 15+: 320730.5673; Demand score: 1.0; Affluence rent: 115000.0; Affluence coverage: 0.9899</small>
- **capture**: Against 63 premium substitutes in a pool of 228 it holds about 1% of premium reviews, in a market with 551 premium reviews per 1k women.  
  <small>Capture: 0.0081; Premium substitutes: 63; Premium salons in catchment: 228; Premium saturation: 550.6052</small>
- **cannibalisation**: 86% of its catchment women are also reached by another lounge.  
  <small>Shared catchment: 0.8553; Cannibalisation score: 0.1447</small>
- **rating**: Its 4.6 rating sits 0.2 below its rivals' 4.8 median.  
  <small>Google rating: 4.6; Substitutes' median rating: 4.8; Rating gap: -0.2</small>

_Each signal is scored 0-1 on a fixed scale where 1 is good for the lounge, and the contribution rows add up to the composite. Premium saturation (premium reviews per 1k women in the catchment) is shown for context and is not scored._

### al-dhafra — **HOLD** (ai)

**HOLD, low confidence: Al Dhafra has its 22,900 women to itself and leads a thin premium market, but the market is too small to PROTECT.**

- **cannibalisation**: No other Bedashing lounge reaches any of its catchment women (0% shared).  
  <small>Shared catchment: 0.0; Cannibalisation score: 1.0; Cannibalisation contribution: 0.2857</small>
- **demand**: Only about 22,900 women live within a 15-min drive, a demand score of 0.11.  
  <small>Addressable women 15+: 22927.3037; Demand score: 0.1146; Demand contribution: 0.0327</small>
- **capture**: It holds 71% of premium reviews, but with only 7 premium salons the market is thin, so its capture score is pulled toward neutral.  
  <small>Capture: 0.7086; Premium salons in catchment: 7; Thin premium market: True; Capture score: 0.85</small>
- **rating**: Its 4.5 rating is 0.2 below its 3 substitutes' 4.7 median.  
  <small>Google rating: 4.5; Substitutes' median rating: 4.7; Rating gap: -0.2; Premium substitutes: 3</small>

_Each signal is scored 0-1 on a fixed scale where 1 is good for the lounge, and the contribution rows add up to the composite. Premium saturation (premium reviews per 1k women in the catchment) is shown for context and is not scored._

...

### khaleej-al-arabi — **SHRINK** (ai)

**SHRINK: 98% of Khaleej Al Arabi's 69,400 catchment women are also reached by a sibling, and its demand score is below neutral.**

- **cannibalisation**: 98% of its catchment women live in cells another Bedashing lounge also reaches.  
  <small>Shared catchment: 0.9766; Cannibalisation score: 0.0234; Cannibalisation contribution: 0.0067</small>
- **demand**: About 69,400 women live within a 15-min drive, a demand score of 0.35.  
  <small>Addressable women 15+: 69366.5903; Demand score: 0.3468; Demand contribution: 0.0991</small>

_Each signal is scored 0-1 on a fixed scale where 1 is good for the lounge, and the contribution rows add up to the composite. Premium saturation (premium reviews per 1k women in the catchment) is shown for context and is not scored._

### mohammed-bin-zayed-city — **SHRINK** (ai)

**SHRINK, low confidence: all of Mohammed Bin Zayed City's 129,000 catchment women are also reached by a sibling, and it holds 4% of premium reviews.**

- **cannibalisation**: 100% of its catchment women live in cells another Bedashing lounge also reaches.  
  <small>Shared catchment: 1.0; Cannibalisation score: 0.0; Cannibalisation contribution: 0.0</small>
- **capture**: Against 18 premium substitutes in a pool of 80 it holds 4% of premium reviews, in a market with 212 premium reviews per 1k women.  
  <small>Capture: 0.0361; Premium substitutes: 18; Premium salons in catchment: 80; Premium saturation: 212.1424</small>
- **rating**: Its 4.4 rating is 0.2 below its rivals' 4.6 median.  
  <small>Google rating: 4.4; Substitutes' median rating: 4.6; Rating gap: -0.2</small>

_Each signal is scored 0-1 on a fixed scale where 1 is good for the lounge, and the contribution rows add up to the composite. Premium saturation (premium reviews per 1k women in the catchment) is shown for context and is not scored._

### noya-plaza — **SHRINK** (ai)

**SHRINK: every one of Noya Plaza's 45,000 catchment women is also reached by a sibling, and it holds 3% of premium reviews.**

- **cannibalisation**: 100% of its catchment women live in cells another Bedashing lounge also reaches, so it scores 0 on cannibalisation.  
  <small>Shared catchment: 1.0; Cannibalisation score: 0.0; Cannibalisation contribution: 0.0</small>
- **capture**: Against 8 premium substitutes in a pool of 38 it holds 3% of premium reviews, from 215 reviews, in a market with 220 premium reviews per 1k women.  
  <small>Capture: 0.0294; Premium substitutes: 8; Premium salons in catchment: 38; Google reviews: 215; Premium saturation: 219.7295</small>
- **demand**: About 45,000 women live within a 15-min drive, a demand score of 0.23.  
  <small>Addressable women 15+: 45036.5814; Demand score: 0.2252; Demand contribution: 0.0643</small>

_Each signal is scored 0-1 on a fixed scale where 1 is good for the lounge, and the contribution rows add up to the composite. Premium saturation (premium reviews per 1k women in the catchment) is shown for context and is not scored._

### westyas — **SHRINK** (ai)

**SHRINK: 98% of Westyas's 42,400 catchment women are also reached by a sibling, and its demand is low.**

- **cannibalisation**: 98% of its catchment women live in cells another Bedashing lounge also reaches.  
  <small>Shared catchment: 0.9823; Cannibalisation score: 0.0177; Cannibalisation contribution: 0.0051</small>
- **demand**: About 42,400 women live within a 15-min drive, a demand score of 0.21.  
  <small>Addressable women 15+: 42368.0713; Demand score: 0.2118; Demand contribution: 0.0605</small>

_Each signal is scored 0-1 on a fixed scale where 1 is good for the lounge, and the contribution rows add up to the composite. Premium saturation (premium reviews per 1k women in the catchment) is shown for context and is not scored._

### zayed-international-airport — **NOT SCORED** (template)

**Not scored: this lounge serves travellers, not the women living around it, so the catchment model doesn't apply.**

- **demand**: Addressable women 15+: 75,830 women, shown but not scored.  
  <small>Addressable women 15+: 75830.1549; Women 15+ in catchment: 75830.1549; Affluence coverage: 0.0; Women captured (est.): 0.0</small>
- **capture**: Capture: 4% of catchment, shown but not scored.  
  <small>Capture: 0.0392; Premium substitutes: 11; Premium salons in catchment: 51; Thin premium market: False</small>

_Each signal is scored 0-1 on a fixed scale, where 1 is good for the lounge, and the composite is their weighted average. Demand counts women within a 15-min drive, weighted by affluence where Dubai rents are known, cannibalisation how many of them another lounge also reaches, capture the lounge's share of premium-salon reviews nearby, and rating its Google rating against those rivals._

The factor table the app shows next to a lounge's pyramid, for one example:

In [5]:
table("lounge", "al-barsha")

,Value,Unit,Threshold,What it means
Factor,,,,
Women 15+ in catchment,"272,371",women,—,"Women aged 15+ living in the ~2 km cells within a 15-min drive (typical midday traffic). WorldPop adults, rebalanced..."
Addressable women 15+,"320,731",women,"GROW needs 20,000","Catchment women weighted by how affluent their cell is (observed median rent vs the women-weighted median, to the af..."
Affluence rent,"115,000",AED/yr,—,"Median annual household rent (DLD Ejari contracts, Jul-Oct 2026), weighted by women, over the cells with an observed..."
Affluence coverage,99%,% of women,"Under 50%: affluence mostly unknown, weighted neutral",Share of the women here living in cells with an observed rent. The rest are weighted neutral: affluence unknown. Dub...
Cells in catchment,64,count,—,Populated ~2 km grid cells whose centre is within the drive time.
Shared catchment,86%,% of catchment women,"Cannibalisation score: 100% of catchment → 0, 0% of catchment → 1; weight 1",Share of the catchment's women who live in cells another open Bedashing lounge also reaches: the cannibalisation mea...
Premium salons in catchment,228,count,—,"Google Places salons in the catchment that count as premium: Google price 'expensive' or above, or (unpriced) at lea..."
Premium saturation,551,reviews per 1k women,Unsaturated under 150,"Premium salons' Google reviews (scaled up for missed salons) per 1,000 women in the searched cells: the same measure..."
Premium substitutes,63,count,—,The most-reviewed premium salons that together hold 60% of the catchment's premium reviews: the lounge's real compet...


## Growth areas: GROW, then WATCH by women

In [6]:
areas = sorted((s for s in subjects if s[0] == "area" and s[2] != "SKIP"),
               key=lambda s: (s[2] != "GROW", -s[3]["women"]))
for k, sid, a, f in areas:
    show(k, sid)

### sharjah-sharjah — **GROW** (ai)

**GROW: this Sharjah area has about 106,000 women and 73 premium reviews per 1k women, passing both tests with room to spare.**

- **size**: About 106,100 addressable women live here, 86,079 above the 20,000 line.  
  <small>Addressable women 15+: 106079.0591; Size line: 20000; Size gap: 86079; Big enough: True</small>
- **saturation**: 26 premium salons give 73 premium reviews per 1k women, 77 under the 150 line.  
  <small>Premium saturation: 72.9351; Premium salons: 26; Saturation line: 150.0; Saturation gap: 77.0649</small>
- **reach**: The nearest lounge, al-jada, is 9.7 km away in a straight line, beyond a 15-min drive.  
  <small>Distance to nearest lounge: 9.6915; Nearest lounge: al-jada</small>

_Size counts women 15+ living here, weighted by affluence where Dubai rents are known; competition counts premium-salon reviews per 1k women in the cells we searched. GROW needs a big enough market and an unsaturated one._

### al-dhaid-sharjah — **GROW** (ai)

**GROW: Al Dhaid has 73,100 women and almost no premium competition, with 5.4 premium reviews per 1k women.**

- **size**: About 73,100 addressable women live here, 53,100 above the 20,000 line.  
  <small>Addressable women 15+: 73067.0302; Size gap: 53067; Size line: 20000</small>
- **saturation**: 8 premium salons give 5.4 premium reviews per 1k women, 145 under the 150 line.  
  <small>Premium saturation: 5.4338; Premium salons: 8; Saturation gap: 144.5662</small>
- **reach**: The nearest lounge, al-jada, is 43.8 km away in a straight line, beyond a 15-min drive.  
  <small>Distance to nearest lounge: 43.7824; Nearest lounge: al-jada</small>

_Size counts women 15+ living here, weighted by affluence where Dubai rents are known; competition counts premium-salon reviews per 1k women in the cells we searched. GROW needs a big enough market and an unsaturated one._

### khor-fakkan-sharjah — **GROW** (ai)

**GROW: Khor Fakkan passes both tests, with 31,200 women and 28 premium reviews per 1k women.**

- **size**: About 31,200 addressable women live here, 11,200 above the 20,000 line.  
  <small>Addressable women 15+: 31162.5661; Size gap: 11163; Size line: 20000</small>
- **saturation**: 11 premium salons give 28 premium reviews per 1k women, 122 under the 150 line.  
  <small>Premium saturation: 28.4775; Premium salons: 11; Saturation gap: 121.5225</small>
- **reach**: The nearest lounge, al-taif-mall, is 26.5 km away in a straight line, beyond a 15-min drive.  
  <small>Distance to nearest lounge: 26.5027; Nearest lounge: al-taif-mall</small>

_Size counts women 15+ living here, weighted by affluence where Dubai rents are known; competition counts premium-salon reviews per 1k women in the cells we searched. GROW needs a big enough market and an unsaturated one._

### kalba-sharjah — **GROW** (ai)

**GROW: Kalba passes both tests, with about 27,600 women and 44 premium reviews per 1k women.**

- **size**: About 27,600 addressable women live here, 7,638 above the 20,000 line.  
  <small>Addressable women 15+: 27637.5444; Size line: 20000; Size gap: 7638; Big enough: True</small>
- **saturation**: 6 premium salons give 44 premium reviews per 1k women, 106 under the 150 line.  
  <small>Premium saturation: 44.0921; Premium salons: 6; Saturation line: 150.0; Saturation gap: 105.9079</small>
- **reach**: The nearest lounge, al-taif-mall, is 12 km away in a straight line, beyond a 15-min drive.  
  <small>Distance to nearest lounge: 12.0006; Nearest lounge: al-taif-mall</small>

_Size counts women 15+ living here, weighted by affluence where Dubai rents are known; competition counts premium-salon reviews per 1k women in the cells we searched. GROW needs a big enough market and an unsaturated one._

### al-jerf-ajman — **GROW** (ai)

**GROW: Al Jerf passes both tests, with 23,900 women and 59 premium reviews per 1k women, though it clears the size line by only 3,851.**

- **size**: 23,851 addressable women live here, 3,851 above the 20,000 line.  
  <small>Addressable women 15+: 23851.187; Size gap: 3851; Size line: 20000</small>
- **saturation**: 11 premium salons give 59 premium reviews per 1k women, 91 under the 150 line.  
  <small>Premium saturation: 59.4423; Premium salons: 11; Saturation gap: 90.5577</small>
- **reach**: The nearest lounge, al-jada, is 13.2 km away in a straight line, beyond a 15-min drive.  
  <small>Distance to nearest lounge: 13.2036; Nearest lounge: al-jada</small>

_Size counts women 15+ living here, weighted by affluence where Dubai rents are known; competition counts premium-salon reviews per 1k women in the cells we searched. GROW needs a big enough market and an unsaturated one._

...

### dibba-al-hisn-sharjah — **WATCH** (ai)

**WATCH: Dibba Al Hisn is under the saturation line, but its 5,300 addressable women are 14,700 short of the 20,000 a lounge needs.**

- **size**: About 5,300 addressable women live here, 14,700 short of the 20,000 line.  
  <small>Addressable women 15+: 5271.8076; Size gap: -14728; Size line: 20000</small>
- **saturation**: 2 premium salons give 91 premium reviews per 1k women, 59 under the 150 line.  
  <small>Premium saturation: 90.533; Premium salons: 2; Saturation gap: 59.467</small>
- **reach**: The nearest lounge, ras-al-khaimah, is 37.9 km away in a straight line, beyond a 15-min drive.  
  <small>Distance to nearest lounge: 37.8936; Nearest lounge: ras-al-khaimah</small>

_Size counts women 15+ living here, weighted by affluence where Dubai rents are known; competition counts premium-salon reviews per 1k women in the cells we searched. GROW needs a big enough market and an unsaturated one._

### al-marmoom-dubai — **WATCH** (ai)

**WATCH: Al Marmoom has little premium competition, but its 5,100 addressable women are 14,900 short of the 20,000 a lounge needs.**

- **size**: About 5,100 addressable women live here, 14,900 short of the 20,000 line, from 5,200 before affluence weighting.  
  <small>Addressable women 15+: 5118.3907; Size gap: -14882; Size line: 20000; Women 15+: 5187.2713; Affluence coverage: 0.1152</small>
- **saturation**: No premium salons were found in the searched cells, so premium saturation is 0, 150 under the line.  
  <small>Premium saturation: 0.0; Premium salons: 0; Saturation gap: 150.0</small>
- **reach**: The nearest lounge, nad-al-sheba, is 22.7 km away in a straight line, beyond a 15-min drive.  
  <small>Distance to nearest lounge: 22.7169; Nearest lounge: nad-al-sheba</small>

_Size counts women 15+ living here, weighted by affluence where Dubai rents are known; competition counts premium-salon reviews per 1k women in the cells we searched. GROW needs a big enough market and an unsaturated one._

### al-muwaihat-ajman — **WATCH** (ai)

**WATCH: Al Muwaihat is close to the saturation line, and its 5,200 addressable women are 14,800 short of the 20,000 a lounge needs.**

- **size**: About 5,200 addressable women live here, 14,800 short of the 20,000 line.  
  <small>Addressable women 15+: 5156.0401; Size gap: -14844; Size line: 20000</small>
- **saturation**: 5 premium salons give 132 premium reviews per 1k women, 18 under the 150 line.  
  <small>Premium saturation: 131.9429; Premium salons: 5; Saturation gap: 18.0571</small>
- **reach**: The nearest lounge, al-jada, is 7.3 km away in a straight line, beyond a 15-min drive.  
  <small>Distance to nearest lounge: 7.2531; Nearest lounge: al-jada</small>

_Size counts women 15+ living here, weighted by affluence where Dubai rents are known; competition counts premium-salon reviews per 1k women in the cells we searched. GROW needs a big enough market and an unsaturated one._

### manama-ajman — **WATCH** (ai)

**WATCH: Manama has little premium competition, but its 5,100 addressable women are 14,900 short of the 20,000 a lounge needs.**

- **size**: About 5,100 addressable women live here, 14,900 short of the 20,000 line.  
  <small>Addressable women 15+: 5146.8426; Size gap: -14853; Size line: 20000</small>
- **saturation**: 1 premium salon gives 0.4 premium reviews per 1k women, far under the 150 line.  
  <small>Premium saturation: 0.3886; Premium salons: 1; Saturation gap: 149.6114</small>
- **reach**: The nearest lounge, al-taif-mall, is 39.3 km away in a straight line, beyond a 15-min drive.  
  <small>Distance to nearest lounge: 39.3036; Nearest lounge: al-taif-mall</small>

_Size counts women 15+ living here, weighted by affluence where Dubai rents are known; competition counts premium-salon reviews per 1k women in the cells we searched. GROW needs a big enough market and an unsaturated one._

### khatt-ras-al-khaimah — **WATCH** (ai)

**WATCH: Khatt has little premium competition, but its 5,100 addressable women are 14,900 short of the 20,000 a lounge needs.**

- **size**: About 5,100 addressable women live here, 14,900 short of the 20,000 line.  
  <small>Addressable women 15+: 5073.2806; Size gap: -14927; Size line: 20000</small>
- **saturation**: No premium salons were found in the searched cells, so premium saturation is 0, 150 under the line.  
  <small>Premium saturation: 0.0; Premium salons: 0; Saturation gap: 150.0</small>
- **reach**: The nearest lounge, ras-al-khaimah, is 14.9 km away in a straight line, beyond a 15-min drive.  
  <small>Distance to nearest lounge: 14.8786; Nearest lounge: ras-al-khaimah</small>
- **data_gap**: Our salon search covered 25% of its women, 25% short of the 50% GROW needs.  
  <small>Competitor data coverage: 0.2496; Coverage needed: 0.5; Coverage gap: -0.2504</small>

_Size counts women 15+ living here, weighted by affluence where Dubai rents are known; competition counts premium-salon reviews per 1k women in the cells we searched. GROW needs a big enough market and an unsaturated one._

In [7]:
table("area", areas[0][1])

,Value,Unit,Threshold,What it means
Factor,,,,
Women 15+,"106,079",women,"SKIP under 5,000",Women aged 15+ living in the area's cells (raw).
Addressable women 15+,"106,079",women,"GROW needs 20,000","Catchment women weighted by how affluent their cell is (observed median rent vs the women-weighted median, to the af..."
Affluence rent,n/a (missing),AED/yr,—,"Median annual household rent (DLD Ejari contracts, Jul-Oct 2026), weighted by women, over the cells with an observed..."
Affluence coverage,0%,% of women,"Under 50%: affluence mostly unknown, weighted neutral",Share of the women here living in cells with an observed rent. The rest are weighted neutral: affluence unknown. Dub...
Cells,53,count,—,Populated ~2 km cells in the area (one contiguous piece).
Worker housing,11%,% of adults,GROW needs under 50%,Share of the area's adults living in worker housing (OSM industrial land use).
Premium salons,26,count,—,Premium salons found in the area's searched cells.
Premium saturation,72.94,reviews per 1k women,Unsaturated under 150,"Premium salons' Google reviews (scaled up for missed salons) per 1,000 women in the searched cells: the same measure..."
Competitor data coverage,97%,% of women,GROW needs 50%,Share of the area's women in cells our salon search covered.


## Play with it

Change a number and see the fallback: the cache key is a hash of the fact sheet, so any change
means the template is served instead of the cached AI text (and `verify` would reject the old
text anyway).

In [8]:
k, sid, a, f = next(s for s in subjects if s[1] == "al-barsha")
changed = {**f, "catchment_women": f["catchment_women"] + 1}
print(explain(k, sid, a, changed, cache=cache).source, "<- template, because one number changed")

template <- template, because one number changed
